In [ ]:
!pip install exifread folium Basemap


 #Loading the dataset
 I stored and uploaded my image datasets of folder_2 in Google Drive as it allows me to connect my Google Drive directly to my Colab environment. The reason is because this approach emulates a production workflow where datasets are stored in a cloud repository rather than on the local machine. This method is the same one I used when working with large image datasets that would otherwise be impractical to upload repeatedly or store locally. It also supports scalability where as datasets grow in size, they can still be accessed without changing the code structure.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Libraries and setup:**
I defined the paths to the dataset folders in Google Drive, specifically for images taken with a 2 second wait time and a 10 second wait time. I chose Folium instead of Matplotlib because this task involves plotting real world GPS coordinates on an actual map background. It saves me time from manually having to project coordinates onto a flat plot.It allow me to easily add markers, tooltips and polylines without extra transformation.

**Extracting GPS coordinates:**
I implemented a function getgps_coordinates() that loops through each image. Within the loop, I filtered for common jpeg and jpg formats before reading the file in binary mode. This is due to EXIF metadata is stored as raw binary data inside the image file. Using exifread, I accessed the GPSLatitude and GPSLongitude tag to retrieve the geographic location stored in the image metadata. The raw GPS values in degrees, minutes, and seconds were converted into decimal coordinates and I adjusted the sign for southern western hemispheres. These coordinates were stored in a list in sequential order.
I called the getgps_coordinates() function for both the 2s wait and 10s wait folders. That produce two separate coordinate list.

**Using Follium Map Library**
Using folium.Map() library, I initialized a map centered on the first coordinate from the 10 second wait dataset, with zoom_start=18 to focus on cone.

**Drawing paths and markers**
To visualize movement, I plotted two polyline paths,blue line for the 2 second images and red line for the 10 second images. Tooltips used for easy identification. For the 2-second and 10 second wait dataset, I went through the list of coordinates one by one. These loops drop pins for each cone location,color coded blue for 2s wait and red for 10s wait. Each labeled with their order so you can tell exactly which cone and wait time each marker represents when you view the map

In [ ]:
import os
import exifread
import folium

# Paths for folder 2 dataset in Google Drive
folder_2 = "/content/drive/MyDrive/Questions_RND/Question2/cones/folder_2"
folder_2s = os.path.join(folder_2, "2s")  #folder for images with 2 second wait time
folder_10s = os.path.join(folder_2, "10s") ##folder for images with 10 second wait time

#  Extracting the GPS coordinate
def getgps_coordinates(folder):
    coordinates = []
    for img in sorted(os.listdir(folder)):
        if img.lower().endswith(('.jpg', '.jpeg')):
            with open(os.path.join(folder, img), 'rb') as f:
                tags = exifread.process_file(f)
                if 'GPS GPSLatitude' in tags:
                    lat = tags['GPS GPSLatitude'].values
                    lon = tags['GPS GPSLongitude'].values
                    lat_ref = tags['GPS GPSLatitudeRef'].values
                    lon_ref = tags['GPS GPSLongitudeRef'].values

                    # Convert GPS values into decimal
                    lat_val = lat[0] + lat[1]/60 + lat[2]/3600
                    lon_val = lon[0] + lon[1]/60 + lon[2]/3600
                    if lat_ref == 'S': lat_val = -lat_val
                    if lon_ref == 'W': lon_val = -lon_val


                    coordinates.append((float(lat_val), float(lon_val)))




    return coordinates

# Get coordinates
coordinates_2s = getgps_coordinates(folder_2s)
coordinates_10s = getgps_coordinates(folder_10s)

# 3. Intilaze map
display = folium.Map(location=coordinates_10s[0], zoom_start=18)

# Add markers and lines
folium.PolyLine(coordinates_2s, color='blue', tooltip='2s wait').add_to(display)
folium.PolyLine(coordinates_10s, color='red', tooltip='10s wait').add_to(display)

for i, (lat, lon) in enumerate(coordinates_2s):
    folium.Marker([lat, lon], icon=folium.Icon(color='blue'), popup=f'2s:Cone {i+1}').add_to(display)

for i, (lat, lon) in enumerate(coordinates_10s):
    folium.Marker([lat, lon], icon=folium.Icon(color='red'), popup=f'10s:Cone{i+1}').add_to(display)

# Save and show map
display


**Conclusion**
Based on observation of the visualisation above,here are the inference and conclusions that can be reached.10s wait provides a significantly more accurate and consistent GPS coordinates.2s wait results are in less precise coordinates with higher degrees of variablity and outliers.If the goal is to have a more reliable cone mapping,using more than 10s stabilization is advisable
